# Scikit-learn: Classical Machine Learning

## Overview
Scikit-learn provides tools for classical machine learning. It includes:
- Regression (linear, polynomial, SVM, etc.)
- Classification (k-NN, decision trees, SVM, etc.)
- Clustering (k-means, DBSCAN, etc.)
- Model evaluation (cross-validation, metrics)

## Why Scikit-learn for Engineering?
- Build surrogate models for CFD
- Classify flow regimes
- Cluster similar simulation results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# 1. Linear Regression
print("=== Linear Regression ===")

# Generate data
np.random.seed(42)
X = np.random.randn(100, 2)
y = 3 * X[:, 0] + 2 * X[:, 1] + 1 + np.random.randn(100) * 0.1

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_:.3f}")
print(f"MSE: {mse:.6f}")
print(f"R²: {r2:.6f}")

# 2. Polynomial Regression
print("\n=== Polynomial Regression ===")

# Generate non-linear data
X_poly = np.random.randn(100, 1)
y_poly = X_poly[:, 0]**2 + np.random.randn(100) * 0.1

# Create a pipeline with polynomial features
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('linear', LinearRegression())
])

# Train
poly_pipeline.fit(X_poly, y_poly)

# Evaluate
y_pred_poly = poly_pipeline.predict(X_poly)
mse_poly = mean_squared_error(y_poly, y_pred_poly)
r2_poly = r2_score(y_poly, y_pred_poly)

print(f"MSE: {mse_poly:.6f}")
print(f"R²: {r2_poly:.6f}")

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(X_poly, y_poly, alpha=0.5, label='Data')
X_plot = np.linspace(-3, 3, 100).reshape(-1, 1)
plt.plot(X_plot, poly_pipeline.predict(X_plot), 'r-', linewidth=2, label='Polynomial Fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Polynomial Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Classification

Classify flow regimes or design configurations.

In [ ]:
# 3. Classification
print("\n=== Classification ===")

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Generate classification data
X_class, y_class = make_classification(n_samples=1000, n_features=5, 
                                        n_informative=3, n_classes=2, 
                                        random_state=42)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_class, y_class, 
                                                     test_size=0.2, random_state=42)

# Train a random forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred_class = clf.predict(X_test)
print(f"Classification Report:")
print(classification_report(y_test, y_pred_class))

# Feature importance
importances = clf.feature_importances_
plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Feature Importance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Engineering Application: CFD Surrogate Model

Build a surrogate model to predict drag coefficient from flow parameters.

In [ ]:
# 4. Engineering Application: CFD Surrogate
print("\n=== CFD Surrogate Model ===")

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# Generate synthetic CFD data
np.random.seed(42)
n_samples = 200

# Parameters: Re, alpha, M
Re = np.random.uniform(1e5, 1e6, n_samples)
alpha = np.random.uniform(0, 20, n_samples)
M = np.random.uniform(0.3, 0.9, n_samples)

# Drag coefficient (simple model)
cd = (0.02 + 
      0.01 * (Re / 1e6) + 
      0.005 * (alpha / 10)**2 + 
      0.02 * (M - 0.5)**2 +
      np.random.randn(n_samples) * 0.001)

# Create feature matrix
X_cfd = np.column_stack([Re, alpha, M])

# Scale features
scaler = StandardScaler()
X_cfd_scaled = scaler.fit_transform(X_cfd)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_cfd_scaled, cd, 
                                                     test_size=0.2, random_state=42)

# Train an SVM regressor
svr = SVR(kernel='rbf', C=100, gamma=0.1)
svr.fit(X_train, y_train)

# Evaluate
y_pred_cfd = svr.predict(X_test)
mse_cfd = mean_squared_error(y_test, y_pred_cfd)
r2_cfd = r2_score(y_test, y_pred_cfd)

print(f"MSE: {mse_cfd:.6f}")
print(f"R²: {r2_cfd:.6f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_cfd, alpha=0.5, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Cd')
plt.ylabel('Predicted Cd')
plt.title('CFD Surrogate Model: Predictions vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. Build a surrogate model for a 5-parameter CFD problem.
2. Compare linear vs polynomial vs SVM regression.
3. Use cross-validation to select the best model.